# Reinforcement Learning for Risk-Based Information Security Contract Review

**A Powers**  
**PhDAI 830 — Reinforcement Learning Project, Part 2**

## Project overview

In this project, I test whether a reinforcement learning agent can make information security contract reviews more consistent. I use **simulated** vendor scenarios instead of actual contracts. My agent recommends ten security controls, decides whether a Vendor Security Assessment (VSA) is needed, and assigns a low, medium, or high risk rating. The final decision would still belong to a human reviewer.

This notebook contains the full implementation, training, held-out evaluation, charts, and a live demonstration. I will use its **actual output** in my slide deck and recorded presentation.

**Run instructions:** In Google Colab, choose **Runtime → Run all**. Training may take several minutes. A results ZIP will be created at the end. Do not present expected results as observed results before running the notebook.

## 0. Install dependencies

I use Python, Gymnasium for the review environment, Stable-Baselines3 for PPO, PyTorch for the neural network, and common plotting/data libraries. This cell installs the packages in the current notebook session.

In [ ]:
%pip -q install "gymnasium>=1.0,<2.0" "stable-baselines3>=2.4,<3.0" "torch>=2.0" "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7"

## 1. Setup and reproducibility

I set random seeds so that my experiment can be repeated more consistently. Small differences can still occur between computing environments.

In [ ]:
import json
import random
import shutil
from itertools import product
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RESULTS = Path('contract_review_results')
RESULTS.mkdir(exist_ok=True)
print('Environment ready')
print('Gymnasium:', gym.__version__)
print('PyTorch:', torch.__version__)

## 2. Scenarios and experimental security baseline

My Part 1 plan identified six input features: contract type, data classification, internal system access, API integration, sensitive-data access, and AI/ML use. It also identified ten security controls and three VSA triggers.

**Change from Part 1:** I had not yet written down an exact control-by-control rule set or risk-scoring formula. I define both below so I can measure the agent's decisions. These rules are **experimental assumptions**, not formal standards or real-world organizational policy.

For VSA, the three triggers are **API access**, **sensitive-data access**, or **AI/ML use**. Risk uses a simple weighted score. A reviewer would need to validate these rules before any real-world use.

All 240 possible combinations are simulated. I reserve 20% as held-out scenarios and never use them for training updates.

In [ ]:
CONTRACT_TYPES = ['MNDA', 'NDA', 'SOW', 'MSA', 'Terms and Conditions']
DATA_CLASSES = ['Public', 'Private', 'Sensitive']
CONTROL_NAMES = [
    'Access control', 'Authentication and MFA', 'Encryption',
    'Data handling', 'Data retention and disposal',
    'Security incident notification', 'Vulnerability management',
    'Logging and monitoring', 'API security', 'AI/ML requirements'
]
RISK_NAMES = ['Low', 'Medium', 'High']

SCENARIOS = list(product(range(5), range(3), range(2), range(2), range(2), range(2)))
random.Random(SEED).shuffle(SCENARIOS)
split = int(0.8 * len(SCENARIOS))
TRAIN_SCENARIOS = SCENARIOS[:split]
TEST_SCENARIOS = SCENARIOS[split:]
assert set(TRAIN_SCENARIOS).isdisjoint(TEST_SCENARIOS)
print(f'{len(SCENARIOS)} simulated scenarios: {len(TRAIN_SCENARIOS)} training, {len(TEST_SCENARIOS)} held out')

def expert_baseline(scenario):
    contract, data, internal, api, sensitive, ai = map(int, scenario)
    handles_data = bool(data > 0 or sensitive)
    service_contract = contract in (2, 3, 4)
    controls = np.array([
        internal or api or sensitive,           # access control
        internal or api or sensitive,           # authentication/MFA
        handles_data or api,                    # encryption
        handles_data,                           # data handling
        handles_data,                           # retention/disposal
        service_contract or handles_data or internal or api or ai,  # incident notice
        internal or api or service_contract,    # vulnerability management
        internal or api or sensitive,           # logging
        api,                                    # API security
        ai                                      # AI/ML requirements
    ], dtype=np.int64)
    vsa = int(bool(api or sensitive or ai))
    risk_score = int(data == 1) + 2*int(data == 2) + 2*internal + 2*api + 2*sensitive + 2*ai
    risk = 0 if risk_score <= 1 else (1 if risk_score <= 4 else 2)
    return controls, vsa, risk

assert len(CONTROL_NAMES) == 10
assert len(SCENARIOS) == 240
assert expert_baseline((0, 0, 0, 0, 0, 0))[1] == 0
assert expert_baseline((0, 0, 0, 1, 0, 0))[1] == 1
print('Experimental baseline defined and basic checks passed.')

## 3. Custom sequential Gymnasium environment

Each episode is one simulated contract review. The agent makes **12 decisions in order**: ten yes/no security controls, one yes/no VSA decision, and one three-level risk decision. The observation contains the six scenario features, the current decision step, and the previous decisions.

My Part 1 reward plan is preserved: **+1** for a correct control decision, **−2** for missing a required control, **−0.5** for adding an unnecessary control, and **+3/−3** for the VSA and risk decisions.

The action space contains three values. For yes/no decisions, action 2 is invalid and receives a penalty. I will report this limitation because a future implementation could use action masking to prevent invalid actions.

In [ ]:
class ContractReviewEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, scenarios, cycle=False):
        super().__init__()
        self.scenarios = list(scenarios)
        if not self.scenarios:
            raise ValueError('At least one scenario is required')
        self.cycle = cycle
        self.cursor = 0
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(30,), dtype=np.float32)
        self.scenario = self.scenarios[0]
        self.step_id = 0
        self.decisions = [-1] * 12

    def _observation(self):
        contract, data, internal, api, sensitive, ai = self.scenario
        features = np.array([contract/4, data/2, internal, api, sensitive, ai], dtype=np.float32)
        stage = np.zeros(12, dtype=np.float32)
        if self.step_id < 12:
            stage[self.step_id] = 1.0
        return np.concatenate([features, stage, np.asarray(self.decisions, dtype=np.float32)])

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if self.cycle:
            self.scenario = self.scenarios[self.cursor % len(self.scenarios)]
            self.cursor += 1
        else:
            self.scenario = self.scenarios[int(self.np_random.integers(len(self.scenarios)))]
        self.step_id = 0
        self.decisions = [-1] * 12
        return self._observation(), {}

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError('Action must be 0, 1, or 2')
        controls, vsa, risk = expert_baseline(self.scenario)
        if self.step_id < 10:
            expected = int(controls[self.step_id])
            reward = -3.0 if action == 2 else (1.0 if action == expected else (-2.0 if expected else -0.5))
        elif self.step_id == 10:
            reward = -3.0 if action == 2 else (3.0 if action == vsa else -3.0)
        else:
            reward = 3.0 if action == risk else -3.0
        self.decisions[self.step_id] = action
        self.step_id += 1
        terminated = self.step_id == 12
        info = {'scenario': self.scenario, 'decisions': self.decisions.copy()} if terminated else {}
        return self._observation(), float(reward), terminated, False, info

check_env(ContractReviewEnv(TRAIN_SCENARIOS), warn=True)
print('Gymnasium environment check passed.')

## 4. Train PPO and record reward history

I use Stable-Baselines3 PPO with a multilayer perceptron policy (`MlpPolicy`). PPO learns a policy from rewards and penalties. I train for **150,000 timesteps** and save the trained model. A callback records episode rewards so I can check whether learning improved.

Training time depends on the Colab session. I will use the measured reward curve rather than assuming convergence.

In [ ]:
class EpisodeRewardHistory(BaseCallback):
    def __init__(self):
        super().__init__()
        self.timesteps = []
        self.rewards = []

    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.timesteps.append(self.num_timesteps)
                self.rewards.append(float(info['episode']['r']))
        return True

TOTAL_TIMESTEPS = 150_000
train_env = Monitor(ContractReviewEnv(TRAIN_SCENARIOS))
agent = PPO('MlpPolicy', train_env, verbose=0, seed=SEED,
            learning_rate=3e-4, n_steps=1024, batch_size=256,
            gamma=0.99, ent_coef=0.005, device='cpu')
history = EpisodeRewardHistory()
agent.learn(total_timesteps=TOTAL_TIMESTEPS, callback=history)
agent.save(str(RESULTS / 'ppo_contract_review'))
assert len(history.rewards) > 0, 'No completed episodes recorded'
print('Training complete:', len(history.rewards), 'episodes')
print('Trained model saved.')

## 5. Training curve and convergence check

I graph a 100-episode rolling average reward. I also compare the early and late portions of training. A rising curve suggests learning, but a flat curve alone does not prove that the agent found an optimal policy.

In [ ]:
reward_data = pd.DataFrame({'timestep': history.timesteps, 'episode_reward': history.rewards})
reward_data['rolling_100'] = reward_data['episode_reward'].rolling(100, min_periods=1).mean()
reward_data.to_csv(RESULTS / 'training_rewards.csv', index=False)

plt.figure(figsize=(10, 5))
plt.plot(reward_data['timestep'], reward_data['rolling_100'], linewidth=1.7)
plt.xlabel('Training timesteps')
plt.ylabel('Average episode reward (last 100 episodes)')
plt.title('PPO Training Reward')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS / 'training_reward_curve.png', dpi=170)
plt.show()

window = min(500, len(reward_data)//3)
early_mean = reward_data['episode_reward'].iloc[:window].mean()
late_mean = reward_data['episode_reward'].iloc[-window:].mean()
print(f'First {window} episodes mean reward: {early_mean:.2f}')
print(f'Last {window} episodes mean reward: {late_mean:.2f}')
print(f'Change: {late_mean - early_mean:+.2f}')

## 6. Evaluate on unseen scenarios

I evaluate the trained PPO policy **only on the 48 held-out scenarios**. I compare it with random decisions and the experimental expert baseline. I measure control accuracy, VSA accuracy, risk accuracy, average reward, missed required controls, and unnecessary controls.

The expert baseline is the **answer key used to create rewards**. Its perfect accuracy is expected, not evidence that it was independently learned. Random performance gives me a basic comparison.

In [ ]:
def run_review(scenario, policy='ppo', rng=None):
    env = ContractReviewEnv([scenario], cycle=True)
    observation, _ = env.reset()
    choices = []
    total_reward = 0.0
    done = False
    while not done:
        step = len(choices)
        if policy == 'ppo':
            action, _ = agent.predict(observation, deterministic=True)
            action = int(action)
        elif policy == 'random':
            if rng is None:
                raise ValueError('Random policy needs a random generator')
            action = int(rng.integers(2 if step < 11 else 3))
        elif policy == 'expert':
            controls, vsa, risk = expert_baseline(scenario)
            action = int(controls[step]) if step < 10 else (int(vsa) if step == 10 else int(risk))
        else:
            raise ValueError('Unknown policy')
        observation, reward, done, _, _ = env.step(action)
        choices.append(action)
        total_reward += reward
    return choices, total_reward

def evaluate(scenarios, policy, seed=123):
    rng = np.random.default_rng(seed)
    rows = []
    for scenario in scenarios:
        choices, reward = run_review(scenario, policy=policy, rng=rng)
        expected_controls, expected_vsa, expected_risk = expert_baseline(scenario)
        predicted_controls = np.asarray(choices[:10])
        rows.append({
            'policy': policy,
            'scenario': str(scenario),
            'reward': reward,
            'control_accuracy': float(np.mean(predicted_controls == expected_controls)),
            'vsa_correct': int(choices[10] == expected_vsa),
            'risk_correct': int(choices[11] == expected_risk),
            'missed_required_controls': int(np.sum((expected_controls == 1) & (predicted_controls != 1))),
            'unnecessary_controls': int(np.sum((expected_controls == 0) & (predicted_controls == 1))),
            'invalid_control_actions': int(np.sum(predicted_controls == 2)),
            'total_required_controls': int(np.sum(expected_controls)),
            'vsa_expected': int(expected_vsa), 'vsa_predicted': int(choices[10]),
            'risk_expected': int(expected_risk), 'risk_predicted': int(choices[11])
        })
    return pd.DataFrame(rows)

results = pd.concat([evaluate(TEST_SCENARIOS, p) for p in ('ppo', 'random', 'expert')], ignore_index=True)
results.to_csv(RESULTS / 'held_out_scenario_results.csv', index=False)
summary = results.groupby('policy', sort=False).agg(
    mean_reward=('reward', 'mean'),
    control_accuracy=('control_accuracy', 'mean'),
    vsa_accuracy=('vsa_correct', 'mean'),
    risk_accuracy=('risk_correct', 'mean'),
    missed_required_controls=('missed_required_controls', 'sum'),
    unnecessary_controls=('unnecessary_controls', 'sum'),
    invalid_control_actions=('invalid_control_actions', 'sum'),
    total_required_controls=('total_required_controls', 'sum')
).reset_index()
summary['required_control_miss_rate'] = summary['missed_required_controls'] / summary['total_required_controls']
summary.to_csv(RESULTS / 'evaluation_summary.csv', index=False)
print(summary.round(3).to_string(index=False))

## 7. Visual comparison of agent performance

These charts are for the results section of my presentation. I will discuss both the accuracy numbers and any missed security controls rather than reporting accuracy alone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric, title in zip(axes,
    ('control_accuracy', 'vsa_accuracy', 'risk_accuracy'),
    ('Security controls', 'VSA decision', 'Risk classification')):
    ax.bar(summary['policy'], summary[metric])
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.set_ylabel('Accuracy')
    for i, value in enumerate(summary[metric]):
        ax.text(i, min(value + 0.02, 1.02), f'{value:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS / 'held_out_accuracy.png', dpi=170)
plt.show()

ppo_summary = summary.loc[summary.policy == 'ppo'].iloc[0]
print('PPO missed required controls:', int(ppo_summary['missed_required_controls']))
print('PPO unnecessary controls:', int(ppo_summary['unnecessary_controls']))
print('PPO invalid control actions:', int(ppo_summary['invalid_control_actions']))

## 8. Live demonstration: two held-out contract reviews

For the recorded demonstration, I will run this cell on screen. It prints the vendor characteristics, the agent's control recommendations, the expected baseline, the VSA decision, and the risk rating. I will point out any differences between the learned policy and the baseline.

In [ ]:
def demonstrate_review(scenario):
    choices, reward = run_review(scenario, policy='ppo')
    controls, vsa, risk = expert_baseline(scenario)
    contract, data, internal, api, sensitive, ai = scenario
    print('=' * 68)
    print('SIMULATED CONTRACT REVIEW')
    print('Contract type:', CONTRACT_TYPES[contract], '| Data:', DATA_CLASSES[data])
    print('Internal access:', bool(internal), '| API:', bool(api),
          '| Sensitive-data access:', bool(sensitive), '| AI/ML:', bool(ai))
    print('-' * 68)
    print(f'{"Security control":35s} {"Agent":10s} {"Baseline":10s}')
    for name, choice, expected in zip(CONTROL_NAMES, choices[:10], controls):
        agent_label = 'Yes' if choice == 1 else ('No' if choice == 0 else 'Invalid')
        print(f'{name:35s} {agent_label:10s} {"Yes" if expected else "No":10s}')
    agent_vsa = 'Yes' if choices[10] == 1 else ('No' if choices[10] == 0 else 'Invalid')
    print('VSA: agent =', agent_vsa, '| baseline =', 'Yes' if vsa else 'No')
    print('Risk: agent =', RISK_NAMES[choices[11]], '| baseline =', RISK_NAMES[risk])
    print('Episode reward:', round(reward, 2))
    print('=' * 68)

print('DEMONSTRATION A')
demonstrate_review(TEST_SCENARIOS[0])
print('\nDEMONSTRATION B')
demonstrate_review(TEST_SCENARIOS[-1])

## 9. Results interpretation and limitations

I will use the printed metrics to discuss what happened, not claim the agent was perfect. I will also explain these limits:

- The data and expected answers are simulated and rule-based. They are not real contract reviews.
- The agent learns a predefined scoring system. It does not independently discover what is secure.
- Only 48 unique combinations are in the held-out test set, and there are only six scenario features.
- Some combinations may be unrealistic, such as sensitive-data access alongside a public-data classification.
- An incorrect recommendation can have different consequences depending on the control. Missing a required control is especially concerning.
- A real deployment would need verified policies, expert review, bias and fairness testing, change controls, and ongoing monitoring.
- Human reviewers remain responsible for the final decision.

**Possible future work:** Add more realistic vendor features, verify rules with reviewers, evaluate multiple seeds, compare other algorithms, use action masking, and test whether the model remains reliable as review rules change.

## 10. Export repository materials and results

I save the trained model, charts, CSVs, a requirements file, and a README with run instructions. The notebook itself is the complete source code and should be uploaded to the repository. I can also upload the standalone `.py` source file supplied with this notebook.

The ZIP created below contains **outputs and documentation**, not the notebook itself. I will upload the notebook separately. If I need to rerun the demonstration without retraining, I can load the saved model using `PPO.load(...)` after defining the environment.

In [ ]:
(RESULTS / 'requirements.txt').write_text(
    'gymnasium>=1.0,<2.0\nstable-baselines3>=2.4,<3.0\ntorch>=2.0\n'
    'numpy>=1.24\npandas>=2.0\nmatplotlib>=3.7\n', encoding='utf-8'
)

readme = """# Reinforcement Learning for Risk-Based Information Security Contract Review

**Author:** A Powers
**Course:** PhDAI 830 — Project Part 2

## Project
A PPO agent learns to make 12 sequential decisions in simulated information security contract reviews: 10 security controls, a Vendor Security Assessment decision, and a risk classification. No real contracts or confidential data are used. The rules are experimental and require human review before any real use.

## Run in Google Colab
1. Upload `A_Powers_RL_ContractReview_Part2.ipynb` to Google Colab.
2. Select Runtime > Run all.
3. Wait for PPO training and evaluation to finish.
4. Download the generated `A_Powers_RL_ContractReview_Part2_results.zip` or the individual files in `contract_review_results`.
5. Use the evaluation table, training curve, and live demonstration cell for the recorded presentation.

## Run locally
Install dependencies from `requirements.txt`, then run the code cells in the notebook in order. The optional standalone `.py` file contains the same implementation and can be run after installing the dependencies.

## Files
- `.ipynb`: complete source, training, evaluation, and demonstration.
- `ppo_contract_review.zip`: trained PPO model.
- `training_rewards.csv` and `training_reward_curve.png`: learning history.
- `held_out_scenario_results.csv`: results for individual test scenarios.
- `evaluation_summary.csv` and `held_out_accuracy.png`: comparison of PPO, random decisions, and the rule baseline.
- `experiment_metadata.json`: settings and experiment notes.
- `requirements.txt`: required Python packages.

## Evaluation design
There are 240 possible simulated scenario combinations. A fixed 80/20 split provides 192 training and 48 held-out combinations. The scoring baseline is written explicitly in the notebook. Perfect baseline scores are expected because it is the answer key, not a trained model. A trained model's accuracy against these rules is not a measure of real-world contract safety.

## Reproducibility
The notebook uses seed 42 and trains for 150,000 timesteps. Results can still vary slightly across computing environments. Use actual generated metrics in any report or presentation.

## Privacy and use
No real vendor data or organization names are included. This is an educational simulation; final security review decisions require human oversight.
"""
(RESULTS / 'README.md').write_text(readme, encoding='utf-8')
metadata = {
    'author': 'A Powers', 'project': 'Risk-Based Information Security Contract Review',
    'seed': SEED, 'total_scenarios': len(SCENARIOS),
    'train_scenarios': len(TRAIN_SCENARIOS), 'held_out_scenarios': len(TEST_SCENARIOS),
    'timesteps_requested': TOTAL_TIMESTEPS,
    'episodes_completed': len(history.rewards),
    'policy': 'PPO MlpPolicy', 'experiment_only': True,
    'part_1_changes': ['Specified control baseline', 'Specified risk score', 'Defined 12-step discrete environment']
}
(RESULTS / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
archive = shutil.make_archive('A_Powers_RL_ContractReview_Part2_results', 'zip', RESULTS)
print('Results ZIP created:', archive)
print('Files included:')
for path in sorted(RESULTS.iterdir()):
    print(' ', path.name)

## 11. Download the results ZIP

This last cell starts a browser download when I run it in Google Colab. If my browser blocks the download, I can use the folder icon in the left sidebar and download the ZIP manually.

In [ ]:
try:
    from google.colab import files
    files.download('A_Powers_RL_ContractReview_Part2_results.zip')
except ImportError:
    print('Not running in Colab. Download the ZIP from the working directory.')

## Presentation and submission checklist

The notebook supplies the evidence needed for the presentation, but **it does not replace the slide deck or recorded video**.

| Required presentation section | What I will show |
|---|---|
| Introduction | Why consistent information security contract reviews matter; Part 1 objective |
| Background | Brief RL/cybersecurity research from Part 1 |
| Methodology | PPO, six features, 12 decisions, reward design |
| Implementation | Gymnasium environment, policy network, training settings, changes from Part 1 |
| Results and analysis | Actual reward curve, held-out accuracy, missed controls, baseline comparison |
| Demonstration | Run Section 8 live with two held-out scenarios |
| Ethical considerations | Simulated data, safety, accountability, privacy, human oversight |
| Conclusion and future work | What worked, what did not, and improvements |

